# Lab 4: Spark associations

*Autor: Adam Jakubowski 193352*

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

In [12]:
from pyspark import SparkContext
import itertools

try:
    sc = SparkContext(appName="Lab4")
    sc.setLogLevel("ERROR")
except:
    pass

file = sc.textFile('data/4.txt')

In [13]:
def split_line(line):
    items = line.split(' ')
    while '' in items:
        items.remove('')
    return items

lines = file.map(lambda line: split_line(line))

for i in lines.take(5):
    print(i)

['FRO11987', 'ELE17451', 'ELE89019', 'SNA90258', 'GRO99222']
['GRO99222', 'GRO12298', 'FRO12685', 'ELE91550', 'SNA11465', 'ELE26917', 'ELE52966', 'FRO90334', 'SNA30755', 'ELE17451', 'FRO84225', 'SNA80192']
['ELE17451', 'GRO73461', 'DAI22896', 'SNA99873', 'FRO86643']
['ELE17451', 'ELE37798', 'FRO86643', 'GRO56989', 'ELE23393', 'SNA11465']
['ELE17451', 'SNA69641', 'FRO86643', 'FRO78087', 'SNA11465', 'GRO39357', 'ELE28573', 'ELE11375', 'DAI54444']


In [15]:
singles = lines.flatMap(lambda x: x).map(lambda x: ((x,), 1))
singles = singles.reduceByKey(lambda x, y: x + y)

filtered_singles = singles.filter(lambda x: x[1] >= 100).collect()
filtered_singles_list = [item[0][0] for item in filtered_singles]

print(f"Found {len(filtered_singles_list)} single objects which occured >= 100 times.")

Found 647 single objects witch occured >= 100 times.


In [20]:
def get_pairs(session_items):
    filtered_items = [item for item in session_items if item in filtered_singles_list]
    return list(itertools.permutations(filtered_items, 2))

pairs = lines.flatMap(get_pairs)
pairs_mapped = pairs.map(lambda x: (x, 1))
pairs_counts = pairs_mapped.reduceByKey(lambda x, y: x + y)
filtered_pairs = pairs_counts.filter(lambda x: x[1] >= 100).collect()

print(f"Found {len(filtered_pairs)} pairs which occured >= 100 times.")
for i, pair_data in enumerate(filtered_pairs[:10]):
    pair = pair_data[0]
    count = pair_data[1]
    print(f"{pair[0]} [{pair[1]}] (count: {count})")

Found 2668 pairs which occured >= 100 times.
ELE17451 [GRO99222] (count: 148)
GRO99222 [ELE17451] (count: 148)
SNA30755 [ELE17451] (count: 111)
ELE17451 [SNA30755] (count: 111)
ELE17451 [DAI22896] (count: 193)
ELE17451 [SNA99873] (count: 270)
DAI22896 [ELE17451] (count: 193)
SNA99873 [ELE17451] (count: 270)
ELE17451 [ELE59935] (count: 181)
ELE17451 [DAI22177] (count: 203)


In [21]:
def get_triples(session_items):
    filtered_items = [item for item in session_items if item in filtered_singles_list]
    return list(itertools.permutations(filtered_items, 3))

triples = lines.flatMap(get_triples)
triples_mapped = triples.map(lambda x: (x, 1))
triples_counts = triples_mapped.reduceByKey(lambda x, y: x + y)
filtered_triples = triples_counts.filter(lambda x: x[1] >= 100).collect()

print(f"Found {len(filtered_triples)} triples which occured >= 100 times.")
for i, triple_data in enumerate(filtered_triples[:15]):
    triple = triple_data[0]
    count = triple_data[1]
    print(f"{triple[0]} {triple[1]} [{triple[2]}] (count: {count})")

sc.stop()

Found 1398 triples which occured >= 100 times.
GRO46854 GRO73461 [DAI75645] (count: 101)
DAI62779 ELE17451 [ELE32164] (count: 277)
ELE17451 ELE32164 [DAI62779] (count: 277)
GRO73461 FRO40251 [GRO69543] (count: 111)
SNA80324 ELE17451 [GRO85051] (count: 158)
DAI75645 FRO40251 [ELE20847] (count: 153)
DAI75645 DAI62779 [GRO85051] (count: 154)
DAI55148 FRO92469 [FRO40251] (count: 105)
ELE17451 GRO59710 [DAI62779] (count: 213)
DAI42493 DAI62779 [ELE92920] (count: 112)
FRO19221 SNA93860 [DAI62779] (count: 125)
DAI62779 ELE17451 [DAI91290] (count: 109)
DAI75645 GRO21487 [FRO40251] (count: 107)
DAI85309 SNA45677 [DAI62779] (count: 118)
SNA80324 FRO40251 [FRO53271] (count: 115)


According to the instructions, I used permutations, which considers the order of elements in pairs and triples. If we had used combinations instead, the order wouldn't matter, and the total count of results would be smaller.